In [ ]:

!pip install pennylane > out


import numpy as np
import pennylane as qml
import matplotlib.pyplot as plt


import numpy as np

def binary_to_integer(vOutput):
    """
    Convert binary vector 'vOutput' to an integer.
    """
    vPows = 2**np.array(range(len(vOutput)))  # Array of powers of 2
    vPows = vPows[::-1]  # Reverse to match binary order
    return np.dot(vOutput, vPows)  # Dot product to convert to an integer

vOutput = np.array((1, 1, 0, 0))
print(binary_to_integer(vOutput))


12


A. Auxillary Block

In [16]:


def set_classical_value_to_register(x, wires):
    """
    Least significant bit is at the rightmost wire.
    """
    bits = [(x >> i) & 1 for i in range(len(wires))][::-1]
    for b, w in zip(bits, wires):
        if b == 1:
            qml.PauliX(wires=w)

def copy_register(src_wires, dst_wires):
    for s, d in zip(src_wires, dst_wires):
        qml.CNOT(wires=[s, d])

def clear_register_with_copy(src_wires, dst_wires):
    for s, d in zip(src_wires, dst_wires):
        qml.CNOT(wires=[s, d])


B. Inplace Quantum-Classical Adder

In [17]:

import pennylane as qml
import numpy as np

def sum_k(k, wires):
    """
    Apply a rotation gate to each wire, by an angle 'k * pi / 2^j'.
    """
    for j in range(len(wires)):
        qml.RZ(k * np.pi / (2**j), wires=wires[j])

def add_in_k_N(k, N, wires_a, wires_aux, is_in_standard_basis=True):
    """
    'wires_a' quantum register modulo "N" for addition of K.
    'wires_aux' auxiliary wire used for intermediate calculations.
    'is_in_standard_basis' (True) standard basis input ; (False) Fourier basis.
    """
    
    
    if is_in_standard_basis:
        qml.QFT(wires=wires_a)
    
    # Step 1: Add 'k' to register
    sum_k(k, wires_a)
    
    # Step 2: Subtract 'N'
    qml.adjoint(sum_k)(N, wires_a)
    
    # Step 3: Conditionally add 'N' back
    qml.adjoint(qml.QFT)(wires=wires_a)
    qml.CNOT(wires=[wires_a[0], wires_aux[0]])  # Most significant bit controls auxiliary bit
    qml.QFT(wires=wires_a)
    
    # Step 4: Add 'N'conditionally
    qml.ctrl(sum_k, control=wires_aux[0])(N, wires_a)
    
    # Step 5: Clear the auxiliary bit
    qml.adjoint(sum_k)(k, wires_a)
    
    # Step 6: Conditionally add 'N' back
    qml.adjoint(qml.QFT)(wires=wires_a)
    qml.PauliX(wires=wires_a[0])
    qml.CNOT(wires=[wires_a[0], wires_aux[0]])  # Most significant bit controls the auxiliary bit
    qml.PauliX(wires=wires_a[0])
    qml.QFT(wires=wires_a)
    
    # Step 7: Add 'k' to register
    sum_k(k, wires_a)
    
    # Return to standard basis if 'is_in_standard_basis' is True
    if is_in_standard_basis:
        qml.adjoint(qml.QFT)(wires=wires_a)

# Create curcuit "cir"
nWireCount = 10
cir = qml.device("default.qubit", wires=nWireCount, shots=1)

@qml.qnode(cir)
def circuit_add_in_k_N(a, k, N, wires_a, wires_aux):
    qml.BasisEmbedding(a, wires=wires_a)
    add_in_k_N(k, N, wires_a, wires_aux)
    return qml.sample()

# Set the wires
wires_a = list(range(1, 4))
wires_aux1 = 0
wires_aux2 = [4]

# Example
N = 7  # Input value between [0, 7]
a = 2  # Input value between [0, N-1]
k = 5  # Input value between [0, N-1]
wires_a = [wires_aux1] + wires_a
vOutput = circuit_add_in_k_N(a, k, N, wires_a, wires_aux2)

print(a, "+", k, "mod", N, "=", binary_to_integer(vOutput[wires_a]))

2 + 5 mod 7 = 0


C. Outplace Quantum-Classic Adder 


In [ ]:
def add_out_k_N(k, N, wires_a, wires_res, wires_aux):
    """
    Result is stored in 'wires_res'.

    k: Classical constant to be added modulo N.
    N: The modulus.
    wires_a: Quantum wires for input.
    wires_res: Quantum wires for result.
    wires_aux: Quantum wires for auxiliary operations.
    """
    if len(wires_aux) < 2:
        print("Not enough ancilla qubits")
        return

    # CNOT
    for i in range(len(wires_res)):
        qml.CNOT(wires=[wires_a[i], wires_res[i]])

    new_wires_res = [wires_aux[0]] + wires_res
    add_in_k_N(k, N, new_wires_res, [wires_aux[1]])

# Circuit "cir"
nWireCount = 8
cir = qml.device("default.qubit", wires=nWireCount, shots=1)

@qml.qnode(cir)
def circuit_add_out_k_N(a, k, N, wires_a, wires_res, wires_aux):
    qml.BasisEmbedding(a, wires=wires_a)
    add_out_k_N(k, N, wires_a, wires_res, wires_aux)
    return qml.sample()

# Set wires
wires_a = list(range(0, 3))
wires_res = list(range(3, 6))
wires_aux = [6, 7]

# Example
N = 7  # Input value between [0, 7]
a = 5  # Input value between [0, N-1]
k = 3  # Input value between [0, N-1]
vOutput = circuit_add_out_k_N(a, k, N, wires_a, wires_res, wires_aux)
print(a, "+", k, "mod", N, "=", binary_to_integer(vOutput[wires_res]))

5 + 3 mod 7 = 1


D. Inplace Quantum-Quantum Adder

In [ ]:

def add_in_N(N, wires_a, wires_b, aux1, aux2):
    """
    Auxiliary wires 'aux1' and 'aux2'.
    N: The modulus.
    wires_a: Quantum wires for the first input.
    wires_b: Quantum wires for the second input.
    aux1: Auxiliary quantum wires.
    aux2: Auxiliary quantum wires.
    """
    new_wires_b = aux1 + wires_b

    # Quantum Fourier transform to 'wires_b'
    qml.QFT(wires=new_wires_b)

    for i in range(len(wires_a)):
        value = 2**(len(wires_a) - 1 - i) % N

        # Add value to wires_b
        qml.ctrl(add_in_k_N, control=wires_a[i])(value, N, new_wires_b, aux2, False)

    # IQFT on 'wires_b'
    qml.adjoint(qml.QFT)(wires=new_wires_b)

# Circuit "cir"
nWireCount = 10
cir = qml.device("default.qubit", wires=nWireCount, shots=1)

@qml.qnode(dev)
def circuit_add_in_N(a, b, N, wires_a, wires_b, aux1, aux2):
    qml.BasisEmbedding(a, wires=wires_a)
    qml.BasisEmbedding(b, wires=wires_b)
    add_in_N(N, wires_a, wires_b, aux1, aux2)
    return qml.sample()

# Set wires
wires_a = list(range(0, 3))
wires_b = list(range(3, 6))
aux1 = [6]
aux2 = [7]

# Example
N = 5  # Input value between [0, 7]
a = 1  # Input value between [0, N-1]
b = 3  # Input value between [0, N-1]
vOutput = circuit_add_in_N(a, b, N, wires_a, wires_b, aux1, aux2)
print(a, "+", b, "mod", N, "=", binary_to_integer(vOutput[wires_b]))

1 + 3 mod 5 = 4


E. Outplace Quantum-Quantum Adder

In [21]:
def add_out_N(N, wires_a, wires_b, wires_res, aux1, aux2):
    """

    N: The modulus.
    wires_a: Quantum wires for the first input.
    wires_b: Quantum wires for the second input.
    wires_res: Quantum wires to store the result.
    aux1: Auxiliary quantum wires.
    aux2: Auxiliary quantum wires.
    """
    # Set 'b' in 'wires_res'
    for i in range(len(wires_b)):
        qml.CNOT(wires=[wires_b[i], wires_res[i]])

    # Sum 'a' to 'wires_res'
    add_in_N(N, wires_a, wires_res, aux1, aux2)

# Circuit "cir"
nWireCount = 11
cir = qml.device("default.qubit", wires=nWireCount, shots=1)

@qml.qnode(cir)
def circuit_add_out_N(a, b, N, wires_a, wires_b, wires_res, aux1, aux2):
    qml.BasisEmbedding(a, wires=wires_a)
    qml.BasisEmbedding(b, wires=wires_b)
    add_out_N(N, wires_a, wires_b, wires_res, aux1, aux2)
    return qml.sample()

# Set wires
wires_a = list(range(0, 3))
wires_b = list(range(3, 6))
wires_res = list(range(6, 9))
aux1 = [9]
aux2 = [10]

# Example
N = 5  # Input value between [0, 7]
a = 1  # Input value between [0, N-1]
b = 3  # Input value between [0, N-1]

vOutput = circuit_add_out_N(a, b, N, wires_a, wires_b, wires_res, aux1, aux2)
print(a, "+", b, "mod", N, "=", binary_to_integer(vOutput[wires_res]))


1 + 3 mod 5 = 4


F. Outplace Quantum-Classic Multiplier

In [22]:
def mult_out_k_N(k, N, wires_a, wires_b, wires_aux):
    """
    k: The classical constant to multiply by.
    N: The modulus.
    wires_a: Quantum wires for the first input.
    wires_b: Quantum wires to store the result.
    wires_aux: Auxiliary quantum wires.
    """
    if len(wires_aux) < 2:
        print("Not enough ancilla qubits")
        return

    # Apply Fourier basis
    new_wires_b = [wires_aux[0]] + wires_b

    qml.QFT(wires=new_wires_b)
    
    for i in range(len(wires_a)):
        value = (k * 2**(len(wires_a) - 1 - i)) % N
        qml.ctrl(add_in_k_N, wires_a[i])(value, N, new_wires_b, [wires_aux[1]], False)

    qml.adjoint(qml.QFT)(wires=new_wires_b)

## Circuit "cir"
nWireCount = 8
cir = qml.device("default.qubit", wires=nWireCount, shots=1)

@qml.qnode(cir)
def circuit_mult_out_k_N(a, b, k, N, wires_a, wires_b, wires_aux):
    qml.BasisEmbedding(a, wires=wires_a)
    qml.BasisEmbedding(b, wires=wires_b)
    mult_out_k_N(k, N, wires_a, wires_b, wires_aux)
    return qml.sample()

## Set wires
wires_a = list(range(0, 3))
wires_b = list(range(3, 6))
wires_aux = [6, 7]

# Example
N = 6  # Input value between [0, 7]
a = 2  # Input value between [0, N-1]
k = 4  # Input value between [0, N-1]
b = 3  # Input value between [0, N-1]

vOutput = circuit_mult_out_k_N(a, b, k, N, wires_a, wires_b, wires_aux)
print(a, "*", k, "+", b, "mod", N, "=", binary_to_integer(vOutput[wires_b]))

2 * 4 + 3 mod 6 = 5


In [ ]:
G. Quantum-classical multiplier

In [23]:
def mult_in_k_N(k, N, wires_a, wires_support, wires_aux):
    """
    k: The classical constant to multiply by.
    N: The modulus.
    wires_a: Quantum wires for the input and to store the result.
    wires_support: Quantum wires for support.
    wires_aux: Auxiliary quantum wires.
    """
    if len(wires_aux) < 2 or len(wires_a) != len(wires_support):
        print("Not enough ancillary qubits")
        return

    if pow(k, N-2, N)*k % N != 1:
        print("does not exist", k, "mod", N)
        return

    # Step 1
    mult_out_k_N(k, N, wires_a, wires_support, wires_aux)

    # Step 2
    for i in range(len(wires_a)):
        qml.SWAP(wires=[wires_a[i], wires_support[i]])

    # Step 3
    inv_k = pow(k, N-2, N)  # modular inverse
    mult_out_k_N(N - inv_k, N, wires_a, wires_support, wires_aux)

## Circuit "cir"
nWireCount = 8
cir = qml.device("default.qubit", wires=nWireCount, shots=1)

@qml.qnode(cir)
def circuit_mult_in_k_N(a, k, N, wires_a, wires_support, wires_aux):
    qml.BasisEmbedding(a, wires=wires_a)
    mult_in_k_N(k, N, wires_a, wires_support, wires_aux)
    return qml.sample()

## Set wires
wires_a = list(range(0, 3))
wires_support = list(range(3, 6))
wires_aux = [6, 7]

# Example
N = 5  # Input value between [0, 7]
a = 1  # Input value between [0, N-1]
k = 3  # Input value between [0, N-1]
vOutput = circuit_mult_in_k_N(a, k, N, wires_a, wires_support, wires_aux)
print(a, "*", k, "mod", N, "=", binary_to_integer(vOutput[wires_a]))

1 * 3 mod 5 = 3


H. Outplace Quantum-Quantum Multiplier

In [24]:
def mult_out_N(N, wires_a, wires_b, wires_res, wires_aux):
    """
    N: The modulus.
    wires_a: Quantum wires for the first input.
    wires_b: Quantum wires for the second input.
    wires_res: Quantum wires to store the result.
    wires_aux: Auxiliary quantum wires.
    """
    #relate auxillary wires to result wires
    new_wires_res = [wires_aux[0]] + wires_res
    
    # QFT
    qml.QFT(wires=new_wires_res)
    
    for i in range(len(wires_a)):
        new_i = len(wires_a) - i - 1
        for j in range(len(wires_b)):
            new_j = len(wires_b) - j - 1
            value = 2**(new_i + new_j) % N
            qml.ctrl(add_in_k_N, control=[wires_a[i], wires_b[j]])(value, N, new_wires_res, [wires_aux[1]], False)
    
    # IQFT
    qml.adjoint(qml.QFT)(wires=new_wires_res)

## Circuit "cir"
nWireCount = 11
cir = qml.device("default.qubit", wires=nWireCount, shots=1)

@qml.qnode(cir)
def circuit_mult_out_N(a, b, N, wires_a, wires_b, wires_res, wires_aux):
    qml.BasisEmbedding(a, wires=wires_a)
    qml.BasisEmbedding(b, wires=wires_b)
    mult_out_N(N, wires_a, wires_b, wires_res, wires_aux)
    return qml.sample()

## Set wires
wires_a = list(range(0, 3))
wires_b = list(range(3, 6))
wires_res = list(range(6, 9))
wires_aux = [9, 10]

# Example
N = 7  # Input value between [0, 7]
a = 3  # Input value between [0, N-1]
b = 5  # Input value between [0, N-1]
vOutput = circuit_mult_out_N(a, b, N, wires_a, wires_b, wires_res, wires_aux)
print(a, "*", b, "mod", N, "=", binary_to_integer(vOutput[wires_res]))

3 * 5 mod 7 = 1


J. Modular Exponential Operator

In [25]:
def Exp_a_N(a, N, wires_x, wires_res, wires_support, wires_aux):
    """
    a: The base of the exponent.
    N: The modulus.
    wires_x: Quantum wires for the input.
    wires_res: Quantum wires to store the result.
    wires_support: Quantum wires for intermediate calculations.
    wires_aux: Auxiliary quantum wires.
    """
    # Set result wires |1>
    qml.PauliX(wires=wires_res[len(wires_res) - 1])
    
    # Mult_in
    n = len(wires_x)
    for i in range(n):
        value = a**(2**(n-i-1))
        qml.ctrl(mult_in_k_N, control=[wires_x[i]])(value, N, wires_res, wires_support, wires_aux)

## Circuit "cir"
nWireCount = 11
dev = qml.device("default.qubit", wires=nWireCount, shots=1)

@qml.qnode(dev)
def circuit_exp_a_N(x, a, N, wires_x, wires_res, wires_support, wires_aux):
    qml.BasisEmbedding(x, wires=wires_x)
    Exp_a_N(a, N, wires_x, wires_res, wires_support, wires_aux)
    return qml.sample()

## Set wires
wires_x = list(range(0, 3))
wires_res = list(range(3, 6))
wires_support = list(range(6, 9))
wires_aux = [9, 10]

# Example
N = 7  # Input value between [0, 7]
x = 3  # Input value between [0, N-1]
a = 2  # Input value between [0, N-1]
vOutput = circuit_exp_a_N(x, a, N, wires_x, wires_res, wires_support, wires_aux)
print(a, "**", x, "mod", N, "=", binary_to_integer(vOutput[wires_res]))
print("The expected result is", a**x % N)


2 ** 3 mod 7 = 1
The expected result is 1


Part 2